In [34]:
import os
import cv2
import numpy as np

def load_dataset_with_motion_only(base_dataset_path, img_size=(64, 64), frame_stride=10):
    X_data = []
    y_labels = []
    
    folder_mappings = {
        'normal': 0,
        'anomaly': 1
    }
    
    for folder_name, assigned_label in folder_mappings.items():
        folder_path = os.path.join(base_dataset_path, folder_name)
            
        video_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.mp4', '.avi', '.mov'))]
        print(f"Processing '{folder_name}' (Label {assigned_label}): Found {len(video_files)} videos.")
        
        for video_name in video_files:
            video_path = os.path.join(folder_path, video_name)
            cap = cv2.VideoCapture(video_path)
            frame_count = 0
            
            back_sub = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=50, detectShadows=False)
            
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                
                if frame_count % frame_stride == 0:
                    fg_mask = back_sub.apply(frame)
                    
                    resized_mask = cv2.resize(fg_mask, img_size)
                    
                    X_data.append(resized_mask)
                    y_labels.append(assigned_label)
                    
                frame_count += 1
            cap.release()
            
    X_data = np.array(X_data, dtype="float32") / 255.0  
    X_data = np.expand_dims(X_data, axis=-1)             
    y_labels = np.array(y_labels, dtype="int32")
    
    print(f" Matrices Shape X: {X_data.shape} | Labels Shape y: {y_labels.shape}")
    
    return X_data, y_labels


DATABASE_PATH = "C:/Users/shekh/SurrakshaSathi" 
X, y = load_dataset_with_motion_only(DATABASE_PATH, img_size=(64, 64), frame_stride=10)

Processing 'normal' (Label 0): Found 15 videos.
Processing 'anomaly' (Label 1): Found 23 videos.
 Matrices Shape X: (9703, 64, 64, 1) | Labels Shape y: (9703,)


In [35]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training Frames (X_train)   : {X_train.shape}")
print(f"Training Labels (y_train)   : {y_train.shape}")
print(f"Validation Frames (X_test)   : {X_test.shape}")
print(f"Validation Labels (y_test)   : {y_test.shape}")

Training Frames (X_train)   : (7762, 64, 64, 1)
Training Labels (y_train)   : (7762,)
Validation Frames (X_test)   : (1941, 64, 64, 1)
Validation Labels (y_test)   : (1941,)


In [36]:
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


C:\Users\shekh\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [37]:
history = model.fit(X_train, y_train,validation_data=(X_test, y_test),epochs=10,batch_size=32)

Epoch 1/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 26s 94ms/step - accuracy: 0.8909 - loss: 0.2659 - val_accuracy: 0.9196 - val_loss: 0.1953
Epoch 2/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 21s 87ms/step - accuracy: 0.9616 - loss: 0.1040 - val_accuracy: 0.9541 - val_loss: 0.1099
Epoch 3/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 43s 94ms/step - accuracy: 0.9794 - loss: 0.0534 - val_accuracy: 0.9629 - val_loss: 0.1109
Epoch 4/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 41s 93ms/step - accuracy: 0.9853 - loss: 0.0363 - val_accuracy: 0.9619 - val_loss: 0.1064
Epoch 5/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 41s 93ms/step - accuracy: 0.9925 - loss: 0.0198 - val_accuracy: 0.9753 - val_loss: 0.0792
Epoch 6/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 42s 96ms/step - accuracy: 0.9921 - loss: 0.0188 - val_accuracy: 0.9789 - val_loss: 0.0713
Epoch 7/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 23s 96ms/step - accuracy: 0.9928 - loss: 0.0189 - val_accuracy: 0.9778 - val_loss: 0.0688
Epoch 8/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 22s 92ms/step - accuracy: 0.9945 - loss: 0.0135 - 

In [42]:
import os
import cv2
import numpy as np

def testing_videos(base_folder_path, trained_model, img_size=(64, 64)):
    if not os.path.exists(base_folder_path):
        print(f"Error: Folder not found at {base_folder_path}")
        return
        
    supported_extensions = ('.mp4', '.avi', '.mov', '.mkv')
    video_files = [f for f in os.listdir(base_folder_path) if f.lower().endswith(supported_extensions)]
    
    if len(video_files) == 0:
        print(f"No videos found in {base_folder_path}")
        return
        
    for vid in video_files:
        full_video_path = os.path.join(base_folder_path, vid)
        cap = cv2.VideoCapture(full_video_path)
        
        native_fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        if native_fps <= 0 or np.isnan(native_fps):
            native_fps = 30.0
            
        target_fps = 3
        frame_stride = max(1, int(round(native_fps / target_fps)))
        
        back_sub = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=50, detectShadows=False)
        video_frames = []
        
        target_indexes = list(range(0, total_frames, frame_stride))
        
        for frame_idx in target_indexes:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()
            if not ret:
                break
                
            fg_mask = back_sub.apply(frame)
            resized = cv2.resize(fg_mask, img_size)
            normalized = resized / 255.0
            video_frames.append(normalized)
            
        cap.release()
        
        if len(video_frames) == 0:
            print(f"Video Name: {vid} -> Error: No frames could be read.")
            print("-" * 50)
            continue
            
        input_batch = np.array(video_frames, dtype="float32")
        input_batch = np.expand_dims(input_batch, axis=-1)
        
        predictions = trained_model(input_batch, training=False).numpy().flatten()
        
        avg_anomaly_score = np.mean(predictions)
        max_anomaly_score = np.max(predictions)
        
        print(f"Video Name: {vid} (Fetched {len(video_frames)} frames)")
        print(f"Average Anomaly Probability: {avg_anomaly_score * 100:.2f}%")
        
        if avg_anomaly_score > 0.5:
            print("Prediction : ANOMALY DETECTED\n\n")
        else:
            print("Prediction : SYSTEM NORMAL\n\n")
        
NEW_TEST_FOLDER = "C:/Users/shekh/SurrakshaSathi/testing_folder"
testing_videos(NEW_TEST_FOLDER, model)

Video Name: Explosion014_x264A.mp4 (Fetched 130 frames directly)
Average Anomaly Probability: 76.10%
Prediction : ANOMALY DETECTED


Video Name: Fighting035_x264A.mp4 (Fetched 97 frames directly)
Average Anomaly Probability: 98.00%
Prediction : ANOMALY DETECTED


Video Name: Fighting037_x264A.mp4 (Fetched 436 frames directly)
Average Anomaly Probability: 84.24%
Prediction : ANOMALY DETECTED


Video Name: Normal_Videos_251_x264.mp4 (Fetched 41 frames directly)
Average Anomaly Probability: 6.21%
Prediction : SYSTEM NORMAL


Video Name: Normal_Videos_360_x264.mp4 (Fetched 99 frames directly)
Average Anomaly Probability: 3.14%
Prediction : SYSTEM NORMAL


Video Name: Normal_Videos_704_x264.mp4 (Fetched 170 frames directly)
Average Anomaly Probability: 1.84%
Prediction : SYSTEM NORMAL


Video Name: Normal_Videos_913_x264.mp4 (Fetched 61 frames directly)
Average Anomaly Probability: 5.89%
Prediction : SYSTEM NORMAL


